# L2C90 · Exact computational verification

**Companion to:** *Characterization of graphs with second largest eigenvalue at most 1 and clique number at least 90*

**Chaohui Chen · Hangxi Cha · Haiying Shan**

This notebook contains the exact computations for Propositions 2.7 and 2.8: enumeration of the minimal forbidden marked graphs and the positive-semidefinite 67-screen. The enumeration and matrix tests are implemented in Python using only the standard library.

Run all cells with a Python 3.10+ kernel to reproduce the complete computation. The accompanying `D.txt` contains the resulting catalogue, so its entries can also be inspected without running the enumeration. The notebook includes saved outputs from a complete execution.


## 1. Definitions

For a marked graph $(H,\boldsymbol c)$ with $\boldsymbol c\in\{0,1\}^{V(H)}$, set

$$
\widehat A(H,\boldsymbol c)=A(H)+\boldsymbol c\boldsymbol c^{\mathsf T}-J,
\qquad M(H,\boldsymbol c)=\widehat A(H,\boldsymbol c)+2I.
$$

The **support graph** of $M$ is the simple graph on the index set of $M$ in which distinct vertices $u,v$ are adjacent if $M_{uv}
e0$.. A marked graph belongs to $\mathcal D$ if $M\not\succeq0$ but every proper principal submatrix of $M$ is positive semidefinite. By the paper's structural result, every member of $\mathcal D$ has connected support and at most 10 vertices. The order bound is a mathematical input to this computation.

The catalogue `D.txt` lists one representative of each marking-preserving isomorphism class as `marking graph6(H)`. Here `graph6` encodes the original graph $H$, **not** its support graph; line numbers start at 1.


## 2. Setup


In [1]:
from __future__ import annotations

import io
import sys
import time
from collections import Counter
from contextlib import redirect_stderr
from fractions import Fraction
from pathlib import Path

if sys.version_info < (3, 10):
    raise RuntimeError('Python 3.10 or later is required.')
if not __debug__:
    raise RuntimeError('Assertions must be enabled; do not use Python -O.')

# Use False to inspect the published catalogue and run the 67-screen only.
RUN_FULL_ENUMERATION = True

ROOT = Path.cwd()
CATALOGUE = ROOT / 'D.txt'
RESULTS = ROOT / 'results'

BOUND = 10

print('Python:', sys.version.split()[0])
print('Full enumeration:', RUN_FULL_ENUMERATION)


Python: 3.13.5
Full enumeration: True


## 3. Exact matrix tests

The matrix $M(H,\boldsymbol c)$ is reconstructed from the marking and support graph. Positive semidefiniteness is tested by exact fraction-free $LDL^{\mathsf T}$ elimination, including singular cases. Determinants are computed by Bareiss elimination. All calculations use integers or rational numbers; no floating-point tolerance is involved.


In [2]:
def matrix_of(bits, rows, n):
    """M = Ahat(H,b) + 2I from the marking and the support graph."""
    a = [[0] * n for _ in range(n)]
    for i in range(n):
        a[i][i] = 1 + ((bits >> i) & 1)
        for j in range(i + 1, n):
            if (rows[i] >> j) & 1:
                a[i][j] = a[j][i] = 1 if ((bits >> i) & 1) and ((bits >> j) & 1) else -1
    return a

def psd(bits, rows, n, skip=-1):
    """True iff M is positive semidefinite on the vertices other than `skip`.

    Exact fraction-free LDL^T (Bareiss).  Each step is taken at a vertex of
    positive diagonal whenever one exists, so the trailing block stays a genuine
    Schur complement up to a positive factor and sign decisions are exact.  A
    remaining block whose diagonal vanishes is positive semidefinite exactly when
    it is zero, for -a_ij^2 would otherwise be a negative 2x2 principal minor.
    """
    verts = [v for v in range(n) if v != skip]
    m = len(verts)
    a = [0] * (m * m)
    for x in range(m):
        i = verts[x]
        bi = (bits >> i) & 1
        a[x * m + x] = 1 + bi
        ri = rows[i]
        for y in range(x + 1, m):
            j = verts[y]
            if (ri >> j) & 1:
                v = 1 if bi and ((bits >> j) & 1) else -1
                a[x * m + y] = v
                a[y * m + x] = v
    previous = 1
    for k in range(m):
        piv = -1
        for i in range(k, m):
            d = a[i * m + i]
            if d < 0:
                return False
            if d > 0 and piv < 0:
                piv = i
        if piv < 0:
            for i in range(k, m):
                for j in range(i + 1, m):
                    if a[i * m + j]:
                        return False
            return True
        if piv != k:
            rk, rp = k * m, piv * m
            a[rk:rk + m], a[rp:rp + m] = a[rp:rp + m], a[rk:rk + m]
            for i in range(m):
                col = i * m
                a[col + k], a[col + piv] = a[col + piv], a[col + k]
        d = a[k * m + k]
        rowk = k * m
        for i in range(k + 1, m):
            base = i * m
            aik = a[base + k]
            for j in range(i, m):
                value, rem = divmod(d * a[base + j] - aik * a[rowk + j], previous)
                if rem:
                    raise ArithmeticError('inexact fraction-free step')
                a[base + j] = value
                a[j * m + i] = value
        previous = d
    return True

def determinant(a):
    """Exact determinant of a square integer matrix (fraction-free, pivoted)."""
    n = len(a)
    a = [row[:] for row in a]
    sign, previous = 1, 1
    for k in range(n - 1):
        piv = next((i for i in range(k, n) if a[i][k]), None)
        if piv is None:
            return 0
        if piv != k:
            a[k], a[piv] = a[piv], a[k]
            sign = -sign
        for i in range(k + 1, n):
            for j in range(k + 1, n):
                value, rem = divmod(a[i][j] * a[k][k] - a[i][k] * a[k][j], previous)
                if rem:
                    raise ArithmeticError('inexact fraction-free step')
                a[i][j] = value
        previous = a[k][k]
    return sign * a[n - 1][n - 1]


## 4. Graph encoding and canonicalization

Support graphs are stored as bit rows. A canonical code identifies marked graphs up to isomorphism preserving the marking. The original graph $H$ is recovered when writing the catalogue.


In [3]:
def support_connected(rows, n):
    if n == 0:
        return True
    seen = 1
    while True:
        more = seen
        for i in range(n):
            if (seen >> i) & 1:
                more |= rows[i]
        if more == seen:
            return seen == (1 << n) - 1
        seen = more

def _refine(rows, partition):
    while True:
        masks = [sum(1 << v for v in cell) for cell in partition]
        split = []
        for cell in partition:
            groups = {}
            for v in cell:
                signature = tuple(bin(rows[v] & mask).count('1') for mask in masks)
                groups.setdefault(signature, []).append(v)
            for key in sorted(groups):
                split.append(tuple(groups[key]))
        if len(split) == len(partition):
            return tuple(split)
        partition = split

def canonical_code(bits, rows):
    """Canonical marks:edges code invariant under marking-preserving isomorphism.

    `edges` is the upper triangle of the *support graph*, not of H.
    """
    n = len(rows)
    if n == 1:
        return '%d:0' % bits
    initial = tuple(tuple(v for v in range(n) if ((bits >> v) & 1) == color)
                    for color in (0, 1))
    initial = tuple(x for x in initial if x)
    best = [None]

    def code(order):
        marks = ''.join('1' if (bits >> v) & 1 else '0' for v in order)
        edges = ''.join('1' if (rows[order[i]] >> order[j]) & 1 else '0'
                        for i in range(n) for j in range(i + 1, n))
        return marks + ':' + edges

    def visit(part):
        part = _refine(rows, part)
        target = next((i for i, cell in enumerate(part) if len(cell) > 1), None)
        if target is None:
            word = code([cell[0] for cell in part])
            if best[0] is None or word < best[0]:
                best[0] = word
            return
        cell = part[target]
        tested = []
        for v in cell:
            # a twin of an already tried vertex yields nothing new
            if any((rows[v] & ~(1 << w)) == (rows[w] & ~(1 << v)) for w in tested):
                continue
            tested.append(v)
            tail = tuple(w for w in cell if w != v)
            visit(part[:target] + ((v,), tail) + part[target + 1:])

    visit(initial)
    return best[0]

def decode(code):
    marks, word = code.split(':')
    n = len(marks)
    bits = sum((c == '1') << i for i, c in enumerate(marks))
    rows = [0] * n
    p = 0
    for i in range(n):
        for j in range(i + 1, n):
            if word[p] == '1':
                rows[i] |= 1 << j
                rows[j] |= 1 << i
            p += 1
    return bits, rows

def edges_of_H(bits, rows):
    """Edges of H recovered from the support graph and the marking."""
    n = len(rows)
    return [(i, j) for i in range(n) for j in range(i + 1, n)
            if ((rows[i] >> j) & 1) == (((bits >> i) & 1) and ((bits >> j) & 1))]


## 5. Recursive enumeration

Start with both markings of a single vertex. Extend every connected-support positive-semidefinite marked graph by one vertex with either marking and any nonempty set of support neighbors. A child is retained as a minimal forbidden marked graph precisely when its full matrix is not positive semidefinite but all vertex-deletion principal submatrices are. Canonicalization removes isomorphic duplicates.

Every connected support graph has a vertex whose removal leaves it connected. Consequently, each minimal forbidden marked graph of order at most 10 can arise from a positive-semidefinite parent in this recursion. Together with the paper's order bound, this gives the exhaustive finite enumeration.


In [4]:
def enumerate_D(progress_every=250):
    """Enumerate connected-support PSD parents and minimal non-PSD children.

    Every member of D has connected support (Theorem 2.6), and a connected
    support graph has a vertex whose deletion keeps it connected. All proper
    principal submatrices of a member of D are PSD. Thus expanding every
    connected-support PSD parent is exhaustive through the order bound 10.
    """
    psd_graphs = {n: set() for n in range(1, BOUND + 1)}
    forbidden = {n: set() for n in range(1, BOUND + 1)}
    psd_graphs[1].add(canonical_code(0, [0]))
    psd_graphs[1].add(canonical_code(1, [0]))
    extensions = 0
    for n in range(1, BOUND):
        todo = sorted(psd_graphs[n])
        terminal = (n + 1 == BOUND)
        for count, parent in enumerate(todo, 1):
            bits, rows = decode(parent)
            for new_mark in (0, 1):
                cbits = bits | (new_mark << n)
                for mask in range(1, 1 << n):
                    crows = [rows[i] | (((mask >> i) & 1) << n)
                             for i in range(n)]
                    crows.append(mask)
                    extensions += 1
                    # Deleting the new vertex gives the PSD parent. Therefore
                    # only the n other vertex deletions have to be checked.
                    if terminal:
                        if all(psd(cbits, crows, n + 1, v) for v in range(n)):
                            if not psd(cbits, crows, n + 1, -1):
                                forbidden[n + 1].add(canonical_code(cbits, crows))
                    elif psd(cbits, crows, n + 1, -1):
                        psd_graphs[n + 1].add(canonical_code(cbits, crows))
                    elif all(psd(cbits, crows, n + 1, v) for v in range(n)):
                        forbidden[n + 1].add(canonical_code(cbits, crows))
            if progress_every and count % progress_every == 0:
                print(f'  processed {count}/{len(todo)} parents of order {n}',
                      file=sys.stderr, flush=True)
        print(f'  completed order {n + 1}', file=sys.stderr, flush=True)
    return forbidden, psd_graphs, extensions


## 6. Exact 67-screen

For each $(H,\boldsymbol c)\in\mathcal D$, test

$$67\bigl(\widehat A(H,\boldsymbol c)+2I\bigr)+2\boldsymbol c\boldsymbol c^{\mathsf T}\succeq0.$$

The test uses exact rational Schur complements, including zero-pivot cases.


In [5]:
def is_psd_exact(matrix):
    """Exact rational Schur-complement PSD check for a symmetric matrix.

    This is separate from the enumerator's fraction-free integer routine.
    If every remaining diagonal entry vanishes, the matrix is PSD if and
    only if the entire remaining matrix is zero.
    """
    a = [[Fraction(x) for x in row] for row in matrix]
    while a:
        n = len(a)
        if any(a[i][i] < 0 for i in range(n)):
            return False
        pivot = next((i for i in range(n) if a[i][i] > 0), None)
        if pivot is None:
            return all(a[i][j] == 0 for i in range(n)
                       for j in range(i + 1, n))
        rest = [i for i in range(n) if i != pivot]
        d = a[pivot][pivot]
        a = [[a[i][j] - a[i][pivot] * a[pivot][j] / d
              for j in rest] for i in rest]
    return True

def passes_67_screen(code):
    """Check 67*(Ahat(H,c)+2I) + 2*c*c^T >= 0 exactly."""
    bits, support = decode(code)
    n = len(support)
    m = matrix_of(bits, support, n)
    b = [(bits >> i) & 1 for i in range(n)]
    screened = [[67 * m[i][j] + 2 * b[i] * b[j]
                 for j in range(n)] for i in range(n)]
    return is_psd_exact(screened)


## 7. Catalogue encoding

The catalogue is written in `marking graph6(H)` format. Reading it back checks the encoding, canonical ordering, marking-preserving uniqueness, connected support, and minimal non-positive-semidefiniteness of every entry.


In [6]:
def graph6_of_H(bits, support):
    """Standard graph6 of H, in the vertex order of a canonical marked graph.

    Graph6 uses edges (0,1), (0,2), (1,2), (0,3), ... , not the
    row-major upper-triangle order used by the internal support code.
    """
    n = len(support)
    if not 1 <= n <= 62:
        raise ValueError('the compact graph6 header supports orders 1..62')
    edge_bits = []
    for j in range(1, n):
        for i in range(j):
            in_support = (support[i] >> j) & 1
            both_marked = ((bits >> i) & 1) and ((bits >> j) & 1)
            edge_bits.append(int(in_support == both_marked))
    edge_bits.extend([0] * ((-len(edge_bits)) % 6))
    packed = ''.join(chr(63 + sum(edge_bits[k + u] << (5 - u)
                                    for u in range(6)))
                     for k in range(0, len(edge_bits), 6))
    return chr(n + 63) + packed

def decode_record(line):
    """Decode one 'marking graph6(H)' record to internal support encoding."""
    columns = line.split()
    if len(columns) != 2:
        raise ValueError('each line must have exactly two fields: marking graph6(H)')
    marks, g6 = columns
    n = len(marks)
    if not 1 <= n <= BOUND or set(marks) - {'0', '1'}:
        raise ValueError('invalid marking')
    if not g6.isascii() or ord(g6[0]) != n + 63:
        raise ValueError('graph6 header/order mismatch')
    nbits = n * (n - 1) // 2
    if len(g6) != 1 + (nbits + 5) // 6:
        raise ValueError('invalid graph6 length')
    raw = []
    for ch in g6[1:]:
        k = ord(ch) - 63
        if not 0 <= k <= 63:
            raise ValueError('invalid graph6 character')
        raw.extend((k >> s) & 1 for s in range(5, -1, -1))
    if any(raw[nbits:]):
        raise ValueError('nonzero graph6 padding')
    bits = sum((ch == '1') << i for i, ch in enumerate(marks))
    support = [0] * n
    p = 0
    for j in range(1, n):
        for i in range(j):
            is_H_edge = raw[p]
            p += 1
            both_marked = ((bits >> i) & 1) and ((bits >> j) & 1)
            if is_H_edge == both_marked:
                support[i] |= 1 << j
                support[j] |= 1 << i
    # Require a normal graph6 representation rather than a hand-written variant.
    if graph6_of_H(bits, support) != g6:
        raise ValueError('graph6 encoding mismatch')
    return bits, support

def write_D(path, forbidden):
    """Write just the graph data: one 'marking graph6(H)' line per graph.

    Entries are ordered by canonical support codes.
    """
    with open(path, 'w', encoding='ascii', newline='\n') as out:
        for n in range(1, BOUND + 1):
            for code in sorted(forbidden[n]):
                bits, support = decode(code)
                marks = ''.join(str((bits >> i) & 1) for i in range(n))
                out.write(marks + ' ' + graph6_of_H(bits, support) + '\n')

def read_D(path):
    """Load D.txt and validate each member, its ordering and its uniqueness."""
    records = []
    seen = set()
    previous = None
    with open(path, 'r', encoding='ascii') as inp:
        for index, line in enumerate(inp, 1):
            if not line.strip():
                raise ValueError(f'empty record at line {index}')
            bits, support = decode_record(line)
            n = len(support)
            code = canonical_code(bits, support)
            if code in seen:
                raise ValueError(f'duplicate marked-graph class at line {index}')
            ordering_key = (n, code)
            if previous is not None and ordering_key <= previous:
                raise ValueError(f'noncanonical ordering at line {index}')
            if not support_connected(support, n):
                raise ValueError(f'disconnected support at line {index}')
            if psd(bits, support, n):
                raise ValueError(f'PSD full matrix at line {index}')
            if any(not psd(bits, support, n, v) for v in range(n)):
                raise ValueError(f'not minimal at line {index}')
            seen.add(code)
            previous = ordering_key
            records.append((index, code, bits, support))
    return records


## 8. Enumeration results (Proposition 2.7)

The complete computation enumerates marked graphs through order 10 and saves the resulting catalogue in `results/D_recomputed.txt`. If `D.txt` is present, the regenerated catalogue is compared with the published data without overwriting it. Set `RUN_FULL_ENUMERATION = False` to read and check `D.txt` without repeating the enumeration.


In [7]:
start = time.perf_counter()
extensions = None
if RUN_FULL_ENUMERATION:
    with redirect_stderr(io.StringIO()):
        forbidden, psd_parents, extensions = enumerate_D(progress_every=0)
    RESULTS.mkdir(exist_ok=True)
    generated = RESULTS / 'D_recomputed.txt'
    write_D(generated, forbidden)
    if CATALOGUE.is_file():
        if generated.read_bytes() != CATALOGUE.read_bytes():
            raise AssertionError('Regenerated catalogue differs from D.txt')
        same_catalogue = True
    else:
        CATALOGUE.write_bytes(generated.read_bytes())
        same_catalogue = None
    data_path = generated
else:
    if not CATALOGUE.is_file():
        raise FileNotFoundError('D.txt is required when RUN_FULL_ENUMERATION is False')
    data_path = CATALOGUE
    same_catalogue = None

records = read_D(data_path)
print('Catalogue entries:', len(records))
if extensions is not None:
    print('Attempted one-vertex extensions:', f'{extensions:,}')
    print('Regenerated catalogue matches D.txt:', 'yes' if same_catalogue else 'new catalogue written')
else:
    print('Catalogue read and validated; full enumeration skipped.')
print('Enumeration and catalogue check (s):', round(time.perf_counter() - start, 2))


Catalogue entries: 1878
Attempted one-vertex extensions: 12,231,262
Regenerated catalogue matches D.txt: yes
Enumeration and catalogue check (s): 444.31


The following table and marking counts are computed from the catalogue obtained above.


In [8]:
counts = Counter(len(rows) for _, _, _, rows in records)
all_marked = sum(bits == (1 << len(rows)) - 1 for _, _, bits, rows in records)
with_unmarked = len(records) - all_marked

print('Order   Minimal forbidden marked graphs')
for n in range(1, BOUND + 1):
    print(f'{n:>5}   {counts[n]:>6,}')
print('Total:                      ', f'{len(records):,}')
print('All marked:                 ', f'{all_marked:,}')
print('With an unmarked vertex:    ', f'{with_unmarked:,}')


Order   Minimal forbidden marked graphs
    1        0
    2        0
    3        4
    4        9
    5       14
    6       18
    7       22
    8       75
    9      323
   10    1,413
Total:                       1,878
All marked:                  1,812
With an unmarked vertex:     66


## 9. The 67-screen (Proposition 2.8)

Apply the screening matrix test to each member of the computed catalogue. Identify the surviving marked graph by comparing it with $H_*=K_1\vee(K_6\cup K_2\cup K_1)$; the graph $H_*$ is specified in the paper.


In [9]:
screen_start = time.perf_counter()
survivors = [(i, code, bits, rows) for i, code, bits, rows in records
             if passes_67_screen(code)]
print('Number screened:', len(records))
print('Number surviving:', len(survivors))
for entry, code, bits, support in survivors:
    print('Catalogue entry:', entry)
    print('Marking:', ''.join(str((bits >> i) & 1) for i in range(len(support))))
    print('graph6(H):', graph6_of_H(bits, support))
print('67-screen time (s):', round(time.perf_counter() - screen_start, 2))


Number screened: 1878
Number surviving: 1
Catalogue entry: 792
Marking: 1111111111
graph6(H): IGCWw{^~w
67-screen time (s): 0.83


The following short exact check identifies the survivor as the fully marked $H_*$ and verifies the identities used in the paper:

$$
\det(2I+A(H_*))=-1,\qquad
(2I+A(H_*))\boldsymbol z=\boldsymbol 1,
\qquad \boldsymbol1^{\mathsf T}\boldsymbol z=-42,
$$

where $\boldsymbol z=(43,-6,-6,-6,-6,-6,-6,-14,-14,-21)^{\mathsf T}$.


In [10]:
HSTAR = 10
HSTAR_EDGES = ([(0, j) for j in range(1, 10)]
               + [(i, j) for i in range(1, 7) for j in range(i + 1, 7)]
               + [(7, 8)])
hstar_support = [0] * HSTAR
for u, v in HSTAR_EDGES:
    hstar_support[u] |= 1 << v
    hstar_support[v] |= 1 << u
hstar_bits = (1 << HSTAR) - 1
hstar_code = canonical_code(hstar_bits, hstar_support)

assert len(survivors) == 1
assert survivors[0][1] == hstar_code
assert survivors[0][2] == hstar_bits
Mstar = matrix_of(hstar_bits, hstar_support, HSTAR)
z = [43] + [-6] * 6 + [-14] * 2 + [-21]
det_mstar = determinant(Mstar)
mz = [sum(a * b for a, b in zip(row, z)) for row in Mstar]
full_psd = psd(hstar_bits, hstar_support, HSTAR)
deletions_psd = all(psd(hstar_bits, hstar_support, HSTAR, v)
                    for v in range(HSTAR))
assert det_mstar == -1
assert mz == [1] * HSTAR
assert sum(z) == -42
assert not full_psd and deletions_psd

print('Identified graph: H* = K1 join (K6 union K2 union K1)')
print('det(2I + A(H*)):', det_mstar)
print('(2I + A(H*)) z = 1:', mz == [1] * HSTAR)
print('1^T z:', sum(z))
print('Every vertex-deletion principal submatrix PSD:', deletions_psd)


Identified graph: H* = K1 join (K6 union K2 union K1)
det(2I + A(H*)): -1
(2I + A(H*)) z = 1: True
1^T z: -42
Every vertex-deletion principal submatrix PSD: True


## Reproducibility

Run all cells with a Python 3.10+ Jupyter kernel in the folder containing `D.txt`. Full enumeration is the default; it generates `results/D_recomputed.txt` and compares it to the published catalogue. For a shorter check of the catalogue and the 67-screen, set `RUN_FULL_ENUMERATION = False`. This shorter run does not regenerate the complete catalogue. All matrix calculations use exact arithmetic.
